In [2]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [176]:
import pandas as pd
import openpyxl
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

In [177]:
# Load the affordability dataset
df_afford = pd.read_excel("data/Affordability Gap Data AY2022-23 2.17.25.xlsx")

# Load the college results dataset
df_results = pd.read_excel("data/College Results View 2021 Data Dump for Export.xlsx")

df_afford.shape, df_results.shape

((21299, 79), (6289, 192))

In [178]:
aff_ids = set(df_afford["Unit ID"].astype(str))
res_ids = set(df_results["UNIQUE_IDENTIFICATION_NUMBER_OF_THE_INSTITUTION"].astype(str))


In [179]:
df = df_afford.merge(
    df_results,
    left_on="Unit ID",
    right_on="UNIQUE_IDENTIFICATION_NUMBER_OF_THE_INSTITUTION",
    how="inner"
)

In [180]:
afford_cols = [
    "Net Price",
    "Affordability Gap (net price minus income earned working 10 hrs at min wage)",
    "Weekly Hours to Close Gap",
    "100% TTD Affordability Gap",
    "150% TTD Affordability Gap",
    "Average Work Study Award",
    "Student Parent Affordability Gap: Center-Based Care",
    "Student Parent Affordability Gap: Home-Based Care",
    "Weekly Hours to Close Gap: Center-Based Care",
    "Weekly Hours to Close Gap: Home-Based Care",
    "Adjusted Annual Center-Based Child Care Cost",
    "Adjusted Annual Home-Based Child Care Cost",
    "Cost of Attendance: Out of State",
    "Cost of Attendance: In State"
]

academic_cols = [
    "Bachelor's Degree Graduation Rate Within 4 Years - Total",
    "Bachelor's Degree Graduation Rate Bachelor Degree Within 6 Years - Total",
    "Bachelor's Degree Graduation Rate Bachelor Degree Within 6 Years - Women",
    "Bachelor's Degree Graduation Rate Within 6 Years - Men",
    "Bachelor's Degree Graduation Rate Within 6 Years - Black, Non-Latino",
    "Bachelor's Degree Graduation Rate Within 6 Years - Latino",
    "Bachelor's Degree Graduation Rate Within 6 Years - Asian",
    "Bachelor's Degree Graduation Rate Within 6 Years - White Non-Latino",
    "Transfer Out Rate",
    "First-Time, Full-Time Retention Rate",
    "Student-to-Faculty Ratio",
    "Total Percent of Applicants Admitted",
    "SAT Evidence Based Reading and Writing - 25th Percentile Score",
    "SAT Evidence Based Reading and Writing - 75th Percentile Score",
    "SAT Math - 25th Percentile Score",
    "SAT Math - 75th Percentile Score",
    "ACT English - 25th Percentile Score",
    "ACT English - 75th Percentile Score",
    "ACT Math - 25th Percentile Score",
    "ACT Math - 75th Percentile Score",
    "Instructional Expenses Per FTE",
    "Instructional Expenses GASB Per Student",
    "Instructional Expenses FASB Per FTE"
]

economic_cols = [
    "Median Earnings of Students Working and Not Enrolled 10 Years After Entry",
    "Median Earnings of Dependent Students Working and Not Enrolled 10 Years After Entry",
    "Median Earnings of Independent Students Working and Not Enrolled 10 Years After Entry",
    "Median Debt for Dependent Students",
    "Median Debt for Independent Students",
    "Median Debt of Completers",
    "Cohort Default Rate",
    "Percent of First-Time, Full-Time Undergraduates Awarded Pell Grants",
]

demo_cols = [
    "Percent of White Undergraduates",
    "Percent of Black or African American Undergraduates",
    "Percent of Latino Undergraduates",
    "Percent of Asian Undergraduates",
    "Percent of American Indian or Alaska Native Undergraduates",
    "Percent of Native Hawaiian or Other Pacific Islander Undergraduates",
    "Percent of Two or More Races Undergraduates",
    "Percent of Women Undergraduates",
    "Percent of Men Undergraduates",
    "Percent of Undergraduates Age 25 to 64",
    "Percent of Undergraduates Enrolled Exclusively in Distance Education Courses",
    "Total Enrollment",
]


degree_cols = [
    "Number of Degrees Awarded in Science, Technology, Engineering, and Math",
    "Number of Degrees Awarded in Arts and Humanities",
    "Number of Degrees Awarded in Education",
    "Number of Degrees Awarded in Social Sciences",
    "Number of Degrees Awarded in Health Sciences",
    "Number of Degrees Awarded in Business",
]


location_cols = [
    "Latitude",
    "Longitude",
    "Region #",
    "Degree of Localization",         # city/suburb/town/rural (numeric)
    "Institution Size Category",      # convert to numeric or one-hot
    "Control of Institution",         # public / private / for-profit
]

msi_cols = [
    "HBCU",
    "HSI",
    "AANAPII",
    "PBI",
    "NANTI",
    "ANNHI",
    "TRIBAL",
]

total = df["Number of Bachelor Degrees Grand Total"].replace(0, np.nan)

engineered = pd.DataFrame({
    "stem_share":        (df["Number of Degrees Awarded in Science, Technology, Engineering, and Math"] / total),
    "business_share":    (df["Number of Degrees Awarded in Business"] / total),
    "health_share":      (df["Number of Degrees Awarded in Health Sciences"] / total),
    "arts_share":        (df["Number of Degrees Awarded in Arts and Humanities"] / total),
    "education_share":   (df["Number of Degrees Awarded in Education"] / total),
    "soc_science_share": (df["Number of Degrees Awarded in Social Sciences"] / total),
}).fillna(0)

df = pd.concat([df, engineered], axis=1)

CURATED_FEATURES = (
    afford_cols
    + academic_cols
    + economic_cols
    + demo_cols
    + degree_cols
    + location_cols
    + msi_cols
    +  [
        # engineered degree-share columns
        "stem_share",
        "business_share",
        "health_share",
        "arts_share",
        "education_share",
        "soc_science_share"
    ]
)
FEATURES = [c for c in CURATED_FEATURES if c in df.columns]

In [181]:
df = df.rename(columns={'Institution Name_x': 'Institution Name'})
df = df.drop(columns=['Institution Name_y'], errors='ignore')
for col in df.columns:
    if df[col].isna().all():
        df[col] = 0
        
# Replace missing numerical values with column medians
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())
df_full = df.copy()


num_cols = df.select_dtypes(include=[np.number]).columns

keep_cols = [c for c in num_cols if df[c].isna().mean() < 0.4]
df = df[keep_cols]

In [182]:
X = df[FEATURES].copy()
X = X.fillna(X.median())

def apply_feature_weights(X, weight_map):
    X_weighted = X.copy()
    for col, w in weight_map.items():
        if col in X_weighted.columns:
            X_weighted[col] = X_weighted[col] * w
    return X_weighted


In [183]:

scaler = StandardScaler()

X_weighted = apply_feature_weights(X, weights)
X_scaled = scaler.fit_transform(X_weighted)
college_embeddings = pca.fit_transform(X_scaled)

df_full["embedding"] = list(college_embeddings)

In [184]:
student = {
    "target_net_price": 15000,
    "max_work_hours": 10,
    "family_income": 45000,
    
    "importance_afford": 0.9,
    "importance_outcomes": 0.6,
    "importance_diversity": 0.5,
    "importance_mobility": 0.8,
    
    "sat_score": 1300,
    "act_score": None,
    
    "self_ethnicity": "Latino",   
    "will_work_job": True,        
    
    "pref_sector": "public",
    "pref_size": "medium",
    
    "home_lat": 37.87,
    "home_lon": -122.26,
    
    "intended_major": "STEM"
}

In [187]:
def build_student_feature_row(student, df_ref, FEATURES):

    baseline = df_ref[FEATURES].median().to_dict()
    row = {col: baseline[col] for col in FEATURES}

    # -------------------------
    # AFFORDABILITY
    # -------------------------
    if "Net Price" in row:
        row["Net Price"] = student["target_net_price"]

    if "Weekly Hours to Close Gap" in row:
        # If they WILL work, allow up to max_work_hours
        # If not, penalize by setting their preferred hours lower
        target_hours = student["max_work_hours"] if student["will_work_job"] else 0
        row["Weekly Hours to Close Gap"] = target_hours

    # Family income affects which affordability row you should have selected EARLIER.
    # That’s already done in your merged df.


    # -------------------------
    # OUTCOMES
    # -------------------------
    grad_col = "Bachelor's Degree Graduation Rate Bachelor Degree Within 6 Years - Total"
    if grad_col in row:
        row[grad_col] = baseline[grad_col] + 25 * student["importance_outcomes"]

    earn_col = "Median Earnings of Students Working and Not Enrolled 10 Years After Entry"
    if earn_col in row:
        row[earn_col] = baseline[earn_col] + 10000 * student["importance_outcomes"]





    # -------------------------
    # STUDENT → LOCATION
    # -------------------------
    if "Latitude" in row:
        row["Latitude"] = student["home_lat"]
    if "Longitude" in row:
        row["Longitude"] = student["home_lon"]


    # -------------------------
    # SECTOR
    # -------------------------
    if "Control of Institution" in row:
        sector_map = {
            "public": 1,
            "private": 2,
            "for-profit": 3
        }
        if student["pref_sector"] in sector_map:
            row["Control of Institution"] = sector_map[student["pref_sector"]]


    # -------------------------
    # SIZE
    # -------------------------
    if "Institution Size Category" in row:
        size_map = {"small": 1, "medium": 2, "large": 3}
        if student["pref_size"] in size_map:
            row["Institution Size Category"] = size_map[student["pref_size"]]


    # -------------------------
    # SAT / ACT
    # -------------------------
    if student["sat_score"]:
        # Rough mappings, can refine
        sat = student["sat_score"]
        row["SAT Evidence Based Reading and Writing - 25th Percentile Score"] = sat * 0.45
        row["SAT Evidence Based Reading and Writing - 75th Percentile Score"] = sat * 0.55
        row["SAT Math - 25th Percentile Score"] = sat * 0.45
        row["SAT Math - 75th Percentile Score"] = sat * 0.55

    if student["act_score"]:
        act = student["act_score"]
        row["ACT Math - 25th Percentile Score"] = act * 0.9
        row["ACT Math - 75th Percentile Score"] = act * 1.1


    # -------------------------
    # MAJOR preference → degree shares
    # -------------------------
    major = student["intended_major"].lower()

    if major == "stem":
        row["stem_share"] = 0.6
    elif major == "business":
        row["business_share"] = 0.6
    elif major == "health":
        row["health_share"] = 0.6
    elif major == "arts":
        row["arts_share"] = 0.6
    elif major == "education":
        row["education_share"] = 0.6
    elif major == "social":
        row["soc_science_share"] = 0.6

    return pd.DataFrame([row])


In [188]:
build_student_feature_row(student, df_full, FEATURES)

,Net Price,Affordability Gap (net price minus income earned working 10 hrs at min wage),Weekly Hours to Close Gap,100% TTD Affordability Gap,150% TTD Affordability Gap,Average Work Study Award,Student Parent Affordability Gap: Center-Based Care,Student Parent Affordability Gap: Home-Based Care,Weekly Hours to Close Gap: Center-Based Care,Weekly Hours to Close Gap: Home-Based Care,...,PBI,NANTI,ANNHI,TRIBAL,stem_share,business_share,health_share,arts_share,education_share,soc_science_share
0,15000,12130.0,10,33135.0,49702.5,2125.0,29673.59,29468.81,64.58,64.25,...,0.0,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0


In [191]:
def match_student(student, df_full, FEATURES, numeric_features, scaler, pca, weights):
    # 1. Build student feature row
    student_df = build_student_feature_row(student, df_full, FEATURES)
    
    # 2. Extract numeric features
    student_numeric = student_df[numeric_features].fillna(df_full[numeric_features].median())

    # 3. Apply feature weights
    student_weighted = apply_feature_weights(student_numeric, weights)

    # 4. Scale using existing scaler
    student_scaled = scaler.transform(student_weighted)

    # 5. Project into PCA space
    student_emb = pca.transform(student_scaled)

    # 6. Cosine similarity with colleges
    sims = cosine_similarity(college_embeddings, student_emb).flatten()

    # 7. Convert to 0–100 score
    df_full["match_score"] = ((sims + 1) / 2) * 100
    
    # 8. Return top 10
    cols_to_show = ["Institution Name", "match_score"]
    return df_full.sort_values("match_score", ascending=False)[cols_to_show].head(10)